# Run Preprocessing for MovieLens 1M

This notebook runs the preprocessing pipeline to prepare data for a Two-Tower recommendation system.

Outputs:
- Encoded ratings
- User features
- User history sequences (rating >= 3)

In [2]:
from preprocessing import Preprocessor
import numpy as np
import pandas as pd
import os

In [3]:
# Initialize the Preprocessor with sequence length of 20
prep = Preprocessor(seq_len=20)
print(f"Preprocessor initialized with seq_len={prep.seq_len}")

Preprocessor initialized with seq_len=20


In [4]:
# Navigate from preprocessing/ folder to dataset/ folder
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else os.getcwd()
project_root = os.path.dirname(notebook_dir)

# Load data from the MovieLens 1M dataset
ratings_path = os.path.join(project_root, "dataset", "ratings.dat")
users_path = os.path.join(project_root, "dataset", "users.dat")

print(f"Loading from: {ratings_path}")
print(f"Loading from: {users_path}")

ratings, users = prep.load_data(ratings_path, users_path)
print(f"Loaded {len(ratings)} ratings and {len(users)} users")
print(f"\nRatings shape: {ratings.shape}")
print(f"Users shape: {users.shape}")

Loading from: /Users/user1/Documents/IT3190E project/dataset/ratings.dat
Loading from: /Users/user1/Documents/IT3190E project/dataset/users.dat
Loaded 1000209 ratings and 6040 users

Ratings shape: (1000209, 4)
Users shape: (6040, 5)


In [5]:
# Encode user and movie IDs into continuous integer ranges
ratings, users = prep.encode_ids(ratings, users)
print(f"Encoded ratings shape: {ratings.shape}")
print(f"Encoded users shape: {users.shape}")
print(f"\nUnique encoded user IDs: {ratings['user_id'].nunique()}")
print(f"Unique encoded movie IDs: {ratings['movie_id'].nunique()}")
print(f"\nSample encoded ratings:\n{ratings.head()}")

Encoded ratings shape: (1000209, 4)
Encoded users shape: (6040, 5)

Unique encoded user IDs: 6040
Unique encoded movie IDs: 3706

Sample encoded ratings:
   user_id  movie_id  rating  timestamp
0     6039       802     4.0  956703932
1     6039      2191     4.0  956703954
2     6039       579     5.0  956703954
3     6039      1839     5.0  956703977
4     6039      1781     4.0  956703977


In [6]:
# Build user history sequences (movies with rating >= 3)
user_hist = prep.build_user_history(ratings)
print(f"Number of users with history: {len(user_hist)}")
print(f"Average history length: {user_hist.apply(len).mean():.2f}")
print(f"Min history length: {user_hist.apply(len).min()}")
print(f"Max history length: {user_hist.apply(len).max()}")
print(f"\nSample user histories (before padding):")
for i in range(min(3, len(user_hist))):
    print(f"  User {i}: {user_hist.iloc[i]}")

Number of users with history: 6039
Average history length: 138.51
Min history length: 1
Max history length: 1968

Sample user histories (before padding):
  User 0: [2969, 957, 1178, 1574, 2147, 1658, 3177, 1117, 2599, 689, 253, 1104, 858, 593, 2488, 1781, 1848, 2889, 877, 1782, 970, 963, 144, 1838, 1025, 853, 1195, 2592, 1154, 2557, 639, 2710, 517, 2898, 2586, 2128, 964, 580, 1107, 2205, 1421, 513, 708, 574, 581, 0, 2483, 740, 2102, 2162, 1727, 1439, 47]
  User 1: [1108, 1127, 1120, 2512, 1201, 2735, 1135, 1104, 309, 2651, 2816, 1765, 1117, 2879, 579, 501, 3235, 1693, 2307, 2821, 106, 1886, 2931, 1155, 2889, 1259, 1106, 1777, 859, 1773, 1656, 1012, 1782, 3412, 3238, 3493, 1167, 1774, 1618, 2523, 1031, 3219, 3107, 2645, 3341, 2853, 258, 2120, 576, 2856, 1161, 1152, 1775, 2046, 3436, 920, 1337, 2013, 2078, 3031, 228, 626, 1024, 1154, 484, 1047, 1414, 1099, 2203, 2166, 2128, 346, 2892, 1173, 3566, 575, 2374, 443, 1848, 466, 157, 1478, 370, 3186, 2708, 1306, 1406, 339, 1826, 2160, 1271, 20

In [7]:
# Pad sequences to fixed length and create final numpy array
user_hist_padded = user_hist.apply(prep.pad_sequence)
user_hist_array = np.stack(user_hist_padded.values)

print(f"User history array created with shape: {user_hist_array.shape}")
print(f"  Expected shape: ({len(user_hist)}, {prep.seq_len})")
print(f"\n Sample padded sequences (first 3 users):")
for i in range(min(3, user_hist_array.shape[0])):
    print(f"  User {i}: {user_hist_array[i]}")

# Summary of all preprocessing outputs
print("\n" + "="*60)
print("PREPROCESSING COMPLETE - Summary of Outputs")
print("="*60)
print(f"1. Encoded Ratings:")
print(f"   - Shape: {ratings.shape}")
print(f"   - Columns: {list(ratings.columns)}")
print(f"\n2. User Features:")
print(f"   - Shape: {users.shape}")
print(f"   - Columns: {list(users.columns)}")
print(f"\n3. User History Sequences (rating >= 3):")
print(f"   - Shape: {user_hist_array.shape}")
print(f"   - Dtype: {user_hist_array.dtype}")

✓ User history array created with shape: (6039, 20)
  Expected shape: (6039, 20)

✓ Sample padded sequences (first 3 users):
  User 0: [2898 2586 2128  964  580 1107 2205 1421  513  708  574  581    0 2483
  740 2102 2162 1727 1439   47]
  User 1: [2296  737 1286  358 2674  445  159 1631  428 1466 2426 1553 3033 1822
  702 1945  283 1737 1550 1420]
  User 2: [1934 2277  982  538  627 2530 1059 2898 3189 1295 2785 1212 1178 1007
 1167 2162 3318 3622  101 1900]

PREPROCESSING COMPLETE - Summary of Outputs
1. Encoded Ratings:
   - Shape: (1000209, 4)
   - Columns: ['user_id', 'movie_id', 'rating', 'timestamp']

2. User Features:
   - Shape: (6040, 5)
   - Columns: ['user_id', 'gender', 'age', 'occupation', 'zip']

3. User History Sequences (rating >= 3):
   - Shape: (6039, 20)
   - Dtype: int64
